In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance

In [2]:
# laod dataset
x = pd.read_csv("/content/X_res.csv")
y = pd.read_csv("/content/y_res.csv")
x.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15


In [4]:
cat_cols = [col for col in x.columns if x[col].dtype == 'object']
num_cols = [col for col in x.columns if x[col].dtype != 'object']
print("numerical cols= ", num_cols, "\n")
print("categorical cols= ", cat_cols)

numerical cols=  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'] 

categorical cols=  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
# split dataset
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.22, random_state=42)
print(X_train.shape, X_test.shape)

(8071, 19) (2277, 19)


In [9]:
y_train.head()

,Churn
5204,0
9500,1
8487,1
724,1
4631,1


In [10]:
# create pipeline but pass the "SeniorCitizen" from encoding
num_pipeline = Pipeline([
    ('std_scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('one_hot_encoder', OneHotEncoder())
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder = "passthrough")

In [11]:
xgb = XGBClassifier(random_state=42, n_jobs=-1, verbose=2)
xgb_pipeline = Pipeline([
    ('full_pipeline', full_pipeline),
    ('xgb', xgb)
]).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:42:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [12]:
# print performance metrics
acc = accuracy_score(y_train, xgb_pipeline.predict(X_train))
precision = precision_score(y_train, xgb_pipeline.predict(X_train))
recall = recall_score(y_train, xgb_pipeline.predict(X_train))
f1 = f1_score(y_train, xgb_pipeline.predict(X_train))
print(f"Accuracy: {acc}\nPrecision: {precision}\nRecall: {recall}\nF1 Score: {f1}")

Accuracy: 0.9433775244703259
Precision: 0.9263233190271817
Recall: 0.963302752293578
F1 Score: 0.9444511972772578


In [ ]:
r = permutation_importance(xgb_pipeline, X_test, y_test,
                           n_repeats=10,
                           random_state=42)

In [ ]:
%pip install eli5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 3.3 MB/s eta 0:00:00


In [ ]:
import eli5
transformed_feature_names = xgb_pipeline.named_steps['full_pipeline'].get_feature_names_out()
eli5.show_weights(xgb_pipeline.named_steps['xgb'], feature_names=transformed_feature_names.tolist())

Weight,Feature
0.4269,cat__Contract_Month-to-month
0.1749,cat__InternetService_Fiber optic
0.0343,cat__OnlineSecurity_No
0.0316,cat__Contract_Two year
0.0241,cat__PhoneService_No
0.0226,cat__StreamingMovies_Yes
0.0207,cat__TechSupport_No
0.0185,cat__Contract_One year
0.0163,num__tenure
0.0142,cat__OnlineBackup_No


In [13]:
import pandas as pd

# Get feature importances from the XGBoost model
feature_importances = xgb_pipeline.named_steps['xgb'].feature_importances_

# Get the transformed feature names from the full_pipeline
transformed_feature_names = xgb_pipeline.named_steps['full_pipeline'].get_feature_names_out()

# Create a DataFrame to display feature importances
importance_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Importance': feature_importances
})

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

display(importance_df)

,Feature,Importance
0,cat__Contract_Month-to-month,0.579686
1,cat__InternetService_Fiber optic,0.055935
2,cat__TechSupport_Yes,0.038786
3,cat__PaymentMethod_Electronic check,0.038061
4,cat__OnlineSecurity_No,0.037117
5,cat__Dependents_No,0.014746
6,cat__StreamingMovies_Yes,0.013583
7,cat__Contract_One year,0.012645
8,cat__TechSupport_No,0.011795
9,cat__StreamingTV_Yes,0.011756


In [15]:
importance_df.to_csv('feature_importances.csv', index=False)